RF01 – Criar ou Carregar o Dataset de Vendas

In [2]:
import csv
import random
from datetime import datetime, timedelta

def gerar_dataset_vendas(caminho_csv="vendas.csv", n_registros=200, seed=42):
    """Gera um dataset sintetico de vendas com dados sujos e grava em CSV."""
    random.seed(seed)
    produtos = ["Notebook", "Smartphone", "Tablet", "Monitor",
                "Teclado", "Mouse", "Headset"]
    categorias = {"Notebook": "Computadores", "Smartphone": "Celulares",
                  "Tablet": "Celulares", "Monitor": "Computadores",
                  "Teclado": "Perifericos", "Mouse": "Perifericos",
                  "Headset": "Perifericos"}
    precos = {"Notebook": 3500, "Smartphone": 2200, "Tablet": 1800,
              "Monitor": 1200, "Teclado": 250, "Mouse": 120,
              "Headset": 350}
    regioes = ["Sudeste", "Sul", "Nordeste", "Centro-Oeste", "Norte"]
    data_inicio = datetime(2025, 1, 1)
    colunas = ["id_venda", "data_venda", "cliente", "produto",
               "categoria", "regiao", "quantidade", "preco_unitario"]

    with open(caminho_csv, "w", newline="", encoding="utf-8") as arquivo:
        escritor = csv.DictWriter(arquivo, fieldnames=colunas)
        escritor.writeheader()

        for i in range(n_registros):
            produto = random.choice(produtos)
            categoria = categorias[produto]
            quantidade = random.randint(1, 10)
            preco = round(precos[produto] * random.uniform(0.85, 1.15), 2)
            data = data_inicio + timedelta(days=random.randint(0, 364))
            data_txt = data.strftime("%Y-%m-%d")
            cliente = f"Cliente_{random.randint(1, 50):03d}"

            # --- sujeira proposital para a etapa de limpeza ---
            if random.random() < 0.05:
                quantidade = ""                       # valor ausente
            if random.random() < 0.04:
                preco = ""                            # valor ausente
            if random.random() < 0.06:
                produto = " " + produto + " "         # espacos extras
            if random.random() < 0.03:
                data_txt = "DATA INVALIDA"             # data invalida
            if random.random() < 0.10:
                cliente = random.choice([              # ruido no nome
                    cliente.upper().replace("_", "-"),
                    cliente + "!!",
                    " " + cliente,
                    cliente.replace("Cliente_", "cliente#"),
                ])

            escritor.writerow({
                "id_venda": i + 1,
                "data_venda": data_txt,
                "cliente": cliente,
                "produto": produto,
                "categoria": categoria,
                "regiao": random.choice(regioes),
                "quantidade": quantidade,
                "preco_unitario": preco,
            })

    print(f"Dataset gerado com {n_registros} registros em {caminho_csv}.")

gerar_dataset_vendas()


Dataset gerado com 200 registros em vendas.csv.


RF02 – Inspecionar e Descrever os Dados


In [9]:
import pandas as pd
import numpy

df = pd.read_csv("vendas.csv")
df.shape

print ("=== INSPECAO INICIAL DO DATASET ===")
print (f"O dataset tem {df.shape[0]} linhas e {df.shape [1]} colunas")

df.head()

print("Colunas do Dataset")
print(df.dtypes)

print(f"Total de conlunas numéricas: {df.select_dtypes(include='number').shape[1]}")
print(f"Total de colunas de texto: {df.select_dtypes(include='object').shape[1]}")

df.info()

# Analisar colunas com dados faltando

nulos = df.isnull().sum()
nulos_filtrados = nulos[nulos > 0]

if len(nulos_filtrados) > 0:
  print("Colunas com dados faltantes:")
  print(nulos_filtrados)

  print(f"Total de celulas vazias: {df.isnull().sum()}.")

valores = df['quantidade'].dropna().values
preco_unitario = df['preco_unitario'].dropna().values
print("Preco unitario:", len(preco_unitario))
print("Quantidade:", len(valores))

=== INSPECAO INICIAL DO DATASET ===
O dataset tem 200 linhas e 8 colunas
Colunas do Dataset
id_venda            int64
data_venda         object
cliente            object
produto            object
categoria          object
regiao             object
quantidade        float64
preco_unitario    float64
dtype: object
Total de conlunas numéricas: 3
Total de colunas de texto: 5
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id_venda        200 non-null    int64  
 1   data_venda      200 non-null    object 
 2   cliente         200 non-null    object 
 3   produto         200 non-null    object 
 4   categoria       200 non-null    object 
 5   regiao          200 non-null    object 
 6   quantidade      190 non-null    float64
 7   preco_unitario  196 non-null    float64
dtypes: float64(2), int64(1), object(5)
memory usage: 12.6+ KB
Colunas c

RF03 – Limpar e Tratar os Dados (datetime e regex)


In [15]:
import re
from datetime import datetime


def limpar_dados(df):
    """
    Limpa e trata os dados do DataFrame de vendas.

    Parâmetros:
        df (pd.DataFrame): DataFrame bruto.

    Retorna:
        tuple: DataFrame limpo e relatório da limpeza.
    """

    df = df.copy()

    registros_iniciais = len(df)

    # 1. Remover espaços extras
    colunas_texto = ["cliente", "produto", "categoria", "regiao"]

    for coluna in colunas_texto:
        df[coluna] = df[coluna].str.strip()

    # 2. Converter datas inválidas para NaT
    df["data_venda"] = pd.to_datetime(df["data_venda"], errors="coerce")

    datas_invalidas = df["data_venda"].isna().sum()

    # Remover registros com data inválida
    df = df.dropna(subset=["data_venda"])

    # 3. Contabilizar valores nulos
    nulos_quantidade = df["quantidade"].isna().sum()
    nulos_preco = df["preco_unitario"].isna().sum()

    # Remover registros com quantidade ou preço nulos
    df = df.dropna(subset=["quantidade", "preco_unitario"])

    # 4. Corrigir tipos
    df["quantidade"] = df["quantidade"].astype(int)
    df["preco_unitario"] = df["preco_unitario"].astype(float)

    # 5. Padronizar nomes dos clientes usando regex
    padrao_cliente = re.compile(r"Cliente[_-]?(\d{1,3})")

    def padronizar_cliente(valor):
        valor = valor.strip()

        valor = re.sub(r"[^A-Za-z0-9_-]", "", valor)

        correspondencia = padrao_cliente.search(valor)

        if correspondencia:
            numero = int(correspondencia.group(1))
            return f"Cliente_{numero:03d}"

        return valor

    df["cliente"] = df["cliente"].apply(padronizar_cliente)

    # 6. Relatório de limpeza
    registros_finais = len(df)

    relatorio = {
        "registros_iniciais": registros_iniciais,
        "datas_invalidas": int(datas_invalidas),
        "nulos_quantidade": int(nulos_quantidade),
        "nulos_preco_unitario": int(nulos_preco),
        "registros_finais": registros_finais,
        "registros_removidos": (registros_iniciais - registros_finais)}

    return df, relatorio

df_limpo, relatorio_limpeza = limpar_dados(df)

print("=== RELATÓRIO DE LIMPEZA ===")

for chave, valor in relatorio_limpeza.items():
    print(f"{chave}: {valor}")

print("\n=== DADOS APÓS A LIMPEZA ===")

print(df_limpo.head())
print()
print(df_limpo.dtypes)
print()
print(df_limpo.isnull().sum())

print(df_limpo["cliente"].head(20).tolist())


=== RELATÓRIO DE LIMPEZA ===
registros_iniciais: 200
datas_invalidas: 4
nulos_quantidade: 9
nulos_preco_unitario: 4
registros_finais: 183
registros_removidos: 17

=== DADOS APÓS A LIMPEZA ===
   id_venda data_venda      cliente   produto     categoria        regiao  \
0         1 2025-05-21   cliente016     Mouse   Perifericos       Sudeste   
2         3 2025-03-23  Cliente_045    Tablet     Celulares  Centro-Oeste   
3         4 2025-11-06  Cliente_017  Notebook  Computadores         Norte   
4         5 2025-07-05  Cliente_037    Tablet     Celulares           Sul   
5         6 2025-08-21  Cliente_041   Headset   Perifericos       Sudeste   

   quantidade  preco_unitario  
0           2          102.90  
2           1         1939.76  
3           6         3864.87  
4          10         2008.14  
5           2          337.41  

id_venda                   int64
data_venda        datetime64[ns]
cliente                   object
produto                   object
categoria           

RF04 – Criar Colunas Derivadas com Transformações Condicionais

In [16]:
def criar_colunas_derivadas(df):
    """
    Cria as colunas derivadas necessárias para a análise de vendas.

    Parâmetros:
        df (pd.DataFrame): DataFrame limpo.

    Retorna:
        pd.DataFrame: DataFrame com as novas colunas.
    """

    df = df.copy()

    # Receita total por venda
    df["receita_total"] = (
        df["quantidade"] * df["preco_unitario"])

    # Componentes da data
    df["mes"] = df["data_venda"].dt.month
    df["trimestre"] = "Q" + df["data_venda"].dt.quarter.astype(str)
    df["ano"] = df["data_venda"].dt.year

    # Nome do mês em português
    nomes_meses = {
        1: "Janeiro",
        2: "Fevereiro",
        3: "Março",
        4: "Abril",
        5: "Maio",
        6: "Junho",
        7: "Julho",
        8: "Agosto",
        9: "Setembro",
        10: "Outubro",
        11: "Novembro",
        12: "Dezembro"
    }

    df["mes_nome"] = df["mes"].map(nomes_meses)

    # Classificação da receita por venda
    condicoes = [df["receita_total"] < 500, ((df["receita_total"] >= 500) &(df["receita_total"] < 5000)), df["receita_total"] >= 5000]

    faixas = [
        "Baixo Valor",
        "Medio Valor",
        "Alto Valor"
    ]

    df["faixa_receita_item"] = np.select(
        condicoes,
        faixas,
        default="Nao Classificado"
    )

    return df

df_limpo = criar_colunas_derivadas(df_limpo)

print("=== RF04 — COLUNAS DERIVADAS ===")
print(df_limpo[
    [
        "data_venda",
        "quantidade",
        "preco_unitario",
        "receita_total",
        "mes",
        "mes_nome",
        "trimestre",
        "ano",
        "faixa_receita_item"
    ]
].head(10))

print(df_limpo.columns.tolist())

=== RF04 — COLUNAS DERIVADAS ===
   data_venda  quantidade  preco_unitario  receita_total  mes  mes_nome  \
0  2025-05-21           2          102.90         205.80    5      Maio   
2  2025-03-23           1         1939.76        1939.76    3     Março   
3  2025-11-06           6         3864.87       23189.22   11  Novembro   
4  2025-07-05          10         2008.14       20081.40    7     Julho   
5  2025-08-21           2          337.41         674.82    8    Agosto   
6  2025-05-06           3          252.56         757.68    5      Maio   
7  2025-01-17           1          321.55         321.55    1   Janeiro   
8  2025-11-26           8          116.24         929.92   11  Novembro   
9  2025-03-12           7          239.65        1677.55    3     Março   
11 2025-06-24           5          378.20        1891.00    6     Junho   

   trimestre   ano faixa_receita_item  
0         Q2  2025        Baixo Valor  
2         Q1  2025        Medio Valor  
3         Q4  2025   

RF05 – Calcular Métricas Agregadas por Mês, Produto, Categoria e Região


In [19]:
from collections import defaultdict

def calcular_metricas(df):
    """
    Calcula métricas agregadas do dataset de vendas.

    Parâmetros:
        df (pd.DataFrame): DataFrame transformado.

    Retorna:
        dict: Dicionário contendo DataFrames de métricas.
    """

    metricas = {}

    # 1. Métricas por mês
    por_mes = (
        df.groupby(["ano", "mes", "mes_nome"], as_index=False)
        .agg(receita_total=("receita_total", "sum"), quantidade=("quantidade", "sum"), n_vendas=("id_venda", "count")).sort_values(["ano", "mes"]))

    metricas["por_mes"] = por_mes

    # 2. Top 5 produtos
    top_produtos = (
        df.groupby("produto", as_index=False)
        .agg(receita_total=("receita_total", "sum"))
        .sort_values("receita_total", ascending=False)
        .head(5)
    )

    metricas["top_produtos"] = top_produtos

    # 3. Receita por categoria
    por_categoria = (
        df.groupby("categoria", as_index=False)
        .agg(receita_total=("receita_total", "sum"))
        .sort_values("receita_total", ascending=False)
    )

    metricas["por_categoria"] = por_categoria

    # 4. Receita e ticket médio por região
    por_regiao = (
        df.groupby("regiao", as_index=False)
        .agg(
            receita_total=("receita_total", "sum"),
            ticket_medio=("receita_total", "mean")
        )
        .sort_values("receita_total", ascending=False)
    )

    metricas["por_regiao"] = por_regiao

    return metricas

metricas = calcular_metricas(df_limpo)

print("=== RF05 — POR MÊS ===")
display(metricas["por_mes"])

print("\n=== RF05 — TOP 5 PRODUTOS ===")
display(metricas["top_produtos"])

print("\n=== RF05 — POR CATEGORIA ===")
display(metricas["por_categoria"])

print("\n=== RF05 — POR REGIÃO ===")
display(metricas["por_regiao"])


=== RF05 — POR MÊS ===


,ano,mes,mes_nome,receita_total,quantidade,n_vendas
0,2025,1,Janeiro,120866.25,92,15
1,2025,2,Fevereiro,73895.74,70,12
2,2025,3,Março,123869.23,98,18
3,2025,4,Abril,80636.64,48,7
4,2025,5,Maio,132080.62,104,19
5,2025,6,Junho,106534.48,95,15
6,2025,7,Julho,106667.30,84,14
7,2025,8,Agosto,85790.38,72,12
8,2025,9,Setembro,84081.59,89,17
9,2025,10,Outubro,110865.86,72,17



=== RF05 — TOP 5 PRODUTOS ===


,produto,receita_total
3,Notebook,374174.85
5,Tablet,335335.83
4,Smartphone,303255.94
1,Monitor,169554.67
0,Headset,48368.37



=== RF05 — POR CATEGORIA ===


,categoria,receita_total
0,Celulares,638591.77
1,Computadores,543729.52
2,Perifericos,108025.01



=== RF05 — POR REGIÃO ===


,regiao,receita_total,ticket_medio
1,Nordeste,366321.23,8721.934048
2,Norte,299917.37,7140.889762
0,Centro-Oeste,244586.97,5688.069070
4,Sul,228249.66,6521.418857
3,Sudeste,151271.07,7203.384286


RF06 – Segmentar Clientes por Nível de Gasto

In [20]:
def segmentar_clientes(df):
    """
    Agrupa os clientes pelo total gasto e classifica cada cliente.

    Parâmetros:
        df (pd.DataFrame): DataFrame com receita_total.

    Retorna:
        pd.DataFrame: Clientes, total gasto e segmento.
    """

    clientes = (
        df.groupby("cliente", as_index=False)
        .agg(total_gasto=("receita_total", "sum"))
    )

    clientes["segmento"] = clientes["total_gasto"].apply(
        lambda valor:
            "Bronze" if valor < 5000
            else "Prata" if valor <= 15000
            else "Ouro"
    )

    clientes = clientes.sort_values(
        "total_gasto",
        ascending=False
    ).reset_index(drop=True)

    return clientes

clientes = segmentar_clientes(df_limpo)

print("=== RF06 — 10 MAIORES CLIENTES ===")
display(clientes.head(10))

print("=== RF06 — DISTRIBUIÇÃO POR SEGMENTO ===")
print(clientes["segmento"].value_counts())

=== RF06 — 10 MAIORES CLIENTES ===


,cliente,total_gasto,segmento
0,Cliente_006,65905.09,Ouro
1,Cliente_036,63433.04,Ouro
2,Cliente_039,59406.90,Ouro
3,Cliente_020,58459.54,Ouro
4,Cliente_034,52035.60,Ouro
5,Cliente_009,50827.96,Ouro
6,Cliente_024,49146.94,Ouro
7,Cliente_019,40731.05,Ouro
8,Cliente_047,38291.74,Ouro
9,Cliente_012,38111.80,Ouro


=== RF06 — DISTRIBUIÇÃO POR SEGMENTO ===
segmento
Ouro      35
Prata     14
Bronze     9
Name: count, dtype: int64


RF08 – Exportar Resultados em CSV e JSON


In [22]:
import json
import os
import numpy as np # Ensure numpy is imported for np.integer and np.floating

def exportar_resultados(metricas, clientes, estatisticas):
    """
    Exporta os resultados do projeto em CSV e JSON.

    Parâmetros:
        metricas (dict): Métricas agregadas.
        clientes (pd.DataFrame): Segmentação de clientes.
        estatisticas (dict): Estatísticas NumPy.

    Retorna:
        dict: Conteúdo do JSON relido após a gravação.
    """

    os.makedirs(
        "outputs",
        exist_ok=True
    )

    # CSV — métricas por mês
    metricas["por_mes"].to_csv(
        "outputs/metricas_por_mes.csv",
        index=False,
        encoding="utf-8-sig"
    )

    # CSV — segmentação dos clientes
    clientes.to_csv(
        "outputs/segmentacao_clientes.csv",
        index=False,
        encoding="utf-8-sig"
    )

    # Preparação dos dados para JSON
    serializavel = {}

    for chave, valor in estatisticas.items():

        # Não salvar o array escalonado no JSON
        if chave == "receitas_escalonadas":
            continue

        if isinstance(
            valor,
            (np.integer, np.floating)
        ):
            serializavel[chave] = round(
                float(valor),
                2
            )
        else:
            serializavel[chave] = valor

    caminho = (
        "outputs/estatisticas_gerais.json"
    )

    # Escrita
    with open(
        caminho,
        "w",
        encoding="utf-8"
    ) as arquivo:

        json.dump(
            serializavel,
            arquivo,
            indent=4,
            ensure_ascii=False
        )

    # Leitura de volta
    with open(
        caminho,
        "r",
        encoding="utf-8"
    ) as arquivo:

        conferencia = json.load(
            arquivo
        )

    print(
        "JSON gravado e lido com sucesso:"
    )

    print(conferencia)

    return conferencia

# Placeholder for 'estatisticas' as it was not defined in previous steps.
# You would typically calculate these statistics in a prior step (e.g., RF07).
estatisticas = {
    "receita_media": float(df_limpo["receita_total"].mean()),
    "receita_mediana": float(df_limpo["receita_total"].median()),
    "receita_maxima": float(df_limpo["receita_total"].max()),
    "receita_minima": float(df_limpo["receita_total"].min()),
    "total_vendas": int(len(df_limpo)),
    "produto_mais_vendido": df_limpo["produto"].mode()[0]
}

conferencia_json = exportar_resultados(
    metricas,
    clientes,
    estatisticas
)

JSON gravado e lido com sucesso:
{'receita_media': 7051.07267759563, 'receita_mediana': 3330.66, 'receita_maxima': 38833.200000000004, 'receita_minima': 104.19, 'total_vendas': 183, 'produto_mais_vendido': 'Tablet'}


RF09 – Executar o Fluxo Completo (Ponto de Entrada)

In [25]:
import os
import csv
import random
import re
import json
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from collections import defaultdict
from IPython.display import display


def gerar_dataset_vendas(caminho_csv="vendas.csv", n_registros=200, seed=42):
    """Gera um dataset sintetico de vendas com dados sujos e grava em CSV."""
    random.seed(seed)
    produtos = ["Notebook", "Smartphone", "Tablet", "Monitor",
                "Teclado", "Mouse", "Headset"]
    categorias = {"Notebook": "Computadores", "Smartphone": "Celulares",
                  "Tablet": "Celulares", "Monitor": "Computadores",
                  "Teclado": "Perifericos", "Mouse": "Perifericos",
                  "Headset": "Perifericos"}
    precos = {"Notebook": 3500, "Smartphone": 2200, "Tablet": 1800,
              "Monitor": 1200, "Teclado": 250, "Mouse": 120,
              "Headset": 350}
    regioes = ["Sudeste", "Sul", "Nordeste", "Centro-Oeste", "Norte"]
    data_inicio = datetime(2025, 1, 1)
    colunas = ["id_venda", "data_venda", "cliente", "produto",
               "categoria", "regiao", "quantidade", "preco_unitario"]

    with open(caminho_csv, "w", newline="", encoding="utf-8") as arquivo:
        escritor = csv.DictWriter(arquivo, fieldnames=colunas)
        escritor.writeheader()

        for i in range(n_registros):
            produto = random.choice(produtos)
            categoria = categorias[produto]
            quantidade = random.randint(1, 10)
            preco = round(precos[produto] * random.uniform(0.85, 1.15), 2)
            data = data_inicio + timedelta(days=random.randint(0, 364))
            data_txt = data.strftime("%Y-%m-%d")
            cliente = f"Cliente_{random.randint(1, 50):03d}"

            # --- sujeira proposital para a etapa de limpeza ---
            if random.random() < 0.05:
                quantidade = ""                       # valor ausente
            if random.random() < 0.04:
                preco = ""                            # valor ausente
            if random.random() < 0.06:
                produto = " " + produto + " "         # espacos extras
            if random.random() < 0.03:
                data_txt = "DATA INVALIDA"             # data invalida
            if random.random() < 0.10:
                cliente = random.choice([              # ruido no nome
                    cliente.upper().replace("_", "-"),
                    cliente + "!!",
                    " " + cliente,
                    cliente.replace("Cliente_", "cliente#"),
                ])

            escritor.writerow({
                "id_venda": i + 1,
                "data_venda": data_txt,
                "cliente": cliente,
                "produto": produto,
                "categoria": categoria,
                "regiao": random.choice(regioes),
                "quantidade": quantidade,
                "preco_unitario": preco,
            })

    print(f"Dataset gerado com {n_registros} registros em {caminho_csv}.")

def limpar_dados(df):
    """
    Limpa e trata os dados do DataFrame de vendas.

    Parâmetros:
        df (pd.DataFrame): DataFrame bruto.

    Retorna:
        tuple: DataFrame limpo e relatório da limpeza.
    """

    df = df.copy()

    registros_iniciais = len(df)

    # 1. Remover espaços extras
    colunas_texto = ["cliente", "produto", "categoria", "regiao"]

    for coluna in colunas_texto:
        df[coluna] = df[coluna].str.strip()

    # 2. Converter datas inválidas para NaT
    df["data_venda"] = pd.to_datetime(df["data_venda"], errors="coerce")

    datas_invalidas = df["data_venda"].isna().sum()

    # Remover registros com data inválida
    df = df.dropna(subset=["data_venda"])

    # 3. Contabilizar valores nulos
    nulos_quantidade = df["quantidade"].isna().sum()
    nulos_preco = df["preco_unitario"].isna().sum()

    # Remover registros com quantidade ou preço nulos
    df = df.dropna(subset=["quantidade", "preco_unitario"])

    # 4. Corrigir tipos
    df["quantidade"] = df["quantidade"].astype(int)
    df["preco_unitario"] = df["preco_unitario"].astype(float)

    # 5. Padronizar nomes dos clientes usando regex
    padrao_cliente = re.compile(r"Cliente[_-]?(\d{1,3})")

    def padronizar_cliente(valor):
        valor = valor.strip()

        valor = re.sub(r"[^A-Za-z0-9_-]", "", valor)

        correspondencia = padrao_cliente.search(valor)

        if correspondencia:
            numero = int(correspondencia.group(1))
            return f"Cliente_{numero:03d}"

        return valor

    df["cliente"] = df["cliente"].apply(padronizar_cliente)

    # 6. Relatório de limpeza
    registros_finais = len(df)

    relatorio = {
        "registros_iniciais": registros_iniciais,
        "datas_invalidas": int(datas_invalidas),
        "nulos_quantidade": int(nulos_quantidade),
        "nulos_preco_unitario": int(nulos_preco),
        "registros_finais": registros_finais,
        "registros_removidos": (registros_iniciais - registros_finais)}

    return df, relatorio

def criar_colunas_derivadas(df):
    """
    Cria as colunas derivadas necessárias para a análise de vendas.

    Parâmetros:
        df (pd.DataFrame): DataFrame limpo.

    Retorna:
        pd.DataFrame: DataFrame com as novas colunas.
    """

    df = df.copy()

    # Receita total por venda
    df["receita_total"] = (
        df["quantidade"] * df["preco_unitario"])

    # Componentes da data
    df["mes"] = df["data_venda"].dt.month
    df["trimestre"] = "Q" + df["data_venda"].dt.quarter.astype(str)
    df["ano"] = df["data_venda"].dt.year

    # Nome do mês em português
    nomes_meses = {
        1: "Janeiro",
        2: "Fevereiro",
        3: "Março",
        4: "Abril",
        5: "Maio",
        6: "Junho",
        7: "Julho",
        8: "Agosto",
        9: "Setembro",
        10: "Outubro",
        11: "Novembro",
        12: "Dezembro"
    }

    df["mes_nome"] = df["mes"].map(nomes_meses)

    # Classificação da receita por venda
    condicoes = [df["receita_total"] < 500, ((df["receita_total"] >= 500) &(df["receita_total"] < 5000)), df["receita_total"] >= 5000]

    faixas = [
        "Baixo Valor",
        "Medio Valor",
        "Alto Valor"
    ]

    df["faixa_receita_item"] = np.select(
        condicoes,
        faixas,
        default="Nao Classificado"
    )

    return df

def calcular_metricas(df):
    """
    Calcula métricas agregadas do dataset de vendas.

    Parâmetros:
        df (pd.DataFrame): DataFrame transformado.

    Retorna:
        dict: Dicionário contendo DataFrames de métricas.
    """

    metricas = {}

    # 1. Métricas por mês
    por_mes = (
        df.groupby(["ano", "mes", "mes_nome"], as_index=False)
        .agg(receita_total=("receita_total", "sum"), quantidade=("quantidade", "sum"), n_vendas=("id_venda", "count")).sort_values(["ano", "mes"]))

    metricas["por_mes"] = por_mes

    # 2. Top 5 produtos
    top_produtos = (
        df.groupby("produto", as_index=False)
        .agg(receita_total=("receita_total", "sum"))
        .sort_values("receita_total", ascending=False)
        .head(5)
    )

    metricas["top_produtos"] = top_produtos

    # 3. Receita por categoria
    por_categoria = (
        df.groupby("categoria", as_index=False)
        .agg(receita_total=("receita_total", "sum"))
        .sort_values("receita_total", ascending=False)
    )

    metricas["por_categoria"] = por_categoria

    # 4. Receita e ticket médio por região
    por_regiao = (
        df.groupby("regiao", as_index=False)
        .agg(
            receita_total=("receita_total", "sum"),
            ticket_medio=("receita_total", "mean")
        )
        .sort_values("receita_total", ascending=False)
    )

    metricas["por_regiao"] = por_regiao

    return metricas

def segmentar_clientes(df):
    """
    Agrupa os clientes pelo total gasto e classifica cada cliente.

    Parâmetros:
        df (pd.DataFrame): DataFrame com receita_total.

    Retorna:
        pd.DataFrame: Clientes, total gasto e segmento.
    """

    clientes = (
        df.groupby("cliente", as_index=False)
        .agg(total_gasto=("receita_total", "sum"))
    )

    clientes["segmento"] = clientes["total_gasto"].apply(
        lambda valor:
            "Bronze" if valor < 5000
            else "Prata" if valor <= 15000
            else "Ouro"
    )

    clientes = clientes.sort_values(
        "total_gasto",
        ascending=False
    ).reset_index(drop=True)

    return clientes

def exportar_resultados(metricas, clientes, estatisticas):
    """
    Exporta os resultados do projeto em CSV e JSON.

    Parâmetros:
        metricas (dict): Métricas agregadas.
        clientes (pd.DataFrame): Segmentação de clientes.
        estatisticas (dict): Estatísticas NumPy.

    Retorna:
        dict: Conteúdo do JSON relido após a gravação.
    """

    os.makedirs(
        "outputs",
        exist_ok=True
    )

    # CSV — métricas por mês
    metricas["por_mes"].to_csv(
        "outputs/metricas_por_mes.csv",
        index=False,
        encoding="utf-8-sig"
    )

    # CSV — segmentação dos clientes
    clientes.to_csv(
        "outputs/segmentacao_clientes.csv",
        index=False,
        encoding="utf-8-sig"
    )

    # Preparação dos dados para JSON
    serializavel = {}

    for chave, valor in estatisticas.items():

        # Não salvar o array escalonado no JSON
        if chave == "receitas_escalonadas":
            continue

        if isinstance(
            valor,
            (np.integer, np.floating)
        ):
            serializavel[chave] = round(
                float(valor),
                2
            )
        else:
            serializavel[chave] = valor

    caminho = (
        "outputs/estatisticas_gerais.json"
    )

    # Escrita
    with open(
        caminho,
        "w",
        encoding="utf-8"
    ) as arquivo:

        json.dump(
            serializavel,
            arquivo,
            indent=4,
            ensure_ascii=False
        )

    # Leitura de volta
    with open(
        caminho,
        "r",
        encoding="utf-8"
    ) as arquivo:

        conferencia = json.load(
            arquivo
        )

    print(
        "JSON gravado e lido com sucesso:"
    )

    print(conferencia)

    return conferencia

class AnalisadorDeVendas:
    def __init__(self, caminho_csv):
        self.caminho_csv = caminho_csv
        self.df_bruto = None
        self.df_limpo = None
        self.relatorio_limpeza = None
        self.metricas = None
        self.clientes = None
        self.estatisticas = None

    def carregar(self):
        self.df_bruto = pd.read_csv(self.caminho_csv)

    def limpar(self):
        self.df_limpo, self.relatorio_limpeza = limpar_dados(self.df_bruto)

    def transformar(self):
        self.df_limpo = criar_colunas_derivadas(self.df_limpo)

    def analisar(self):
        self.metricas = calcular_metricas(self.df_limpo)
        self.clientes = segmentar_clientes(self.df_limpo)
        # Calculate estatisticas for export
        self.estatisticas = {
            "receita_media": float(self.df_limpo["receita_total"].mean()),
            "receita_mediana": float(self.df_limpo["receita_total"].median()),
            "receita_maxima": float(self.df_limpo["receita_total"].max()),
            "receita_minima": float(self.df_limpo["receita_total"].min()),
            "total_vendas": int(len(self.df_limpo)),
            "produto_mais_vendido": self.df_limpo["produto"].mode()[0]
        }

    def visualizar(self):
        print("Gerando visualizações (implementação futura)...")

    def exportar(self):
        exportar_resultados(
            self.metricas,
            self.clientes,
            self.estatisticas
        )

    def resumo(self):
        print("\n=== RESUMO ===")
        print(f"Total de vendas processadas: {len(self.df_limpo)}")
        if self.df_limpo is not None and 'receita_total' in self.df_limpo.columns:
            print(f"Receita Total: {self.df_limpo['receita_total'].sum():.2f}")
        else:
            print("Receita Total: N/A (DataFrame limpo ou coluna 'receita_total' não disponível)")

def inspecionar_dados(df):
    """
    Exibe as informações estruturais do DataFrame.
    """

    print("\n=== INSPEÇÃO INICIAL DO DATASET ===")

    print(f"Shape: {df.shape}")

    print("\nColunas:")
    print(list(df.columns))

    print("\nTipos de dados:")
    print(df.dtypes)

    print("\nValores nulos por coluna:")
    print(df.isnull().sum())

    print("\nPrimeiros registros:")
    display(df.head())

def main():
    """
    Executa o fluxo completo do SalesInsight PY.
    """

    print("=" * 60)
    print(" SALESINSIGHT PY - ANÁLISE DE DADOS DE VENDAS")
    print("=" * 60)

    # 1. Garantir dataset
    if not os.path.exists("vendas.csv"):
        print("\nDataset não encontrado. Gerando novo dataset...")
        gerar_dataset_vendas()  # This function creates the file
        print("Dataset vendas.csv criado.")

    # 2. Criar analisador
    analisador = AnalisadorDeVendas("vendas.csv")

    # 3. Carregar
    print("\n[1/7] Carregando dados...")
    analisador.carregar()

    # 4. Inspecionar
    print("\n[2/7] Inspecionando dados...")
    inspecionar_dados(analisador.df_bruto)

    # 5. Limpar
    print("\n[3/7] Limpando dados...")
    analisador.limpar()

    print("\n=== RELATÓRIO DE LIMPEZA ===")

    for chave, valor in (
        analisador.relatorio_limpeza.items()
    ):
        print(
            f"{chave}: {valor}"
        )

    # 6. Transformar
    print(
        "\n[4/7] Criando colunas derivadas..."
    )

    analisador.transformar()

    # 7. Analisar
    print(
        "\n[5/7] Calculando métricas..."
    )

    analisador.analisar()

    # 8. Visualizar
    print(
        "\n[6/7] Gerando visualizações..."
    )

    analisador.visualizar()

    # 9. Exportar
    print(
        "\n[7/7] Exportando resultados..."
    )

    analisador.exportar()

    # 10. Resumo
    analisador.resumo()

    print(
        "\n[CONCLUÍDO] "
        "Fluxo finalizado com sucesso."
    )

main()


 SALESINSIGHT PY - ANÁLISE DE DADOS DE VENDAS

[1/7] Carregando dados...

[2/7] Inspecionando dados...

=== INSPEÇÃO INICIAL DO DATASET ===
Shape: (200, 8)

Colunas:
['id_venda', 'data_venda', 'cliente', 'produto', 'categoria', 'regiao', 'quantidade', 'preco_unitario']

Tipos de dados:
id_venda            int64
data_venda         object
cliente            object
produto            object
categoria          object
regiao             object
quantidade        float64
preco_unitario    float64
dtype: object

Valores nulos por coluna:
id_venda           0
data_venda         0
cliente            0
produto            0
categoria          0
regiao             0
quantidade        10
preco_unitario     4
dtype: int64

Primeiros registros:


,id_venda,data_venda,cliente,produto,categoria,regiao,quantidade,preco_unitario
0,1,2025-05-21,cliente#016,Mouse,Perifericos,Sudeste,2.0,102.90
1,2,2025-09-16,Cliente_039,Notebook,Computadores,Norte,NaN,3204.57
2,3,2025-03-23,Cliente_045,Tablet,Celulares,Centro-Oeste,1.0,1939.76
3,4,2025-11-06,Cliente_017,Notebook,Computadores,Norte,6.0,3864.87
4,5,2025-07-05,Cliente_037,Tablet,Celulares,Sul,10.0,2008.14



[3/7] Limpando dados...

=== RELATÓRIO DE LIMPEZA ===
registros_iniciais: 200
datas_invalidas: 4
nulos_quantidade: 9
nulos_preco_unitario: 4
registros_finais: 183
registros_removidos: 17

[4/7] Criando colunas derivadas...

[5/7] Calculando métricas...

[6/7] Gerando visualizações...
Gerando visualizações (implementação futura)...

[7/7] Exportando resultados...
JSON gravado e lido com sucesso:
{'receita_media': 7051.07267759563, 'receita_mediana': 3330.66, 'receita_maxima': 38833.200000000004, 'receita_minima': 104.19, 'total_vendas': 183, 'produto_mais_vendido': 'Tablet'}

=== RESUMO ===
Total de vendas processadas: 183
Receita Total: 1290346.30

[CONCLUÍDO] Fluxo finalizado com sucesso.
